# Deep Agents - Quickstart


## 1. Deep Agents란?

`Deep Agents`는 `LangGraph` 기반으로 구축된 고급 AI 에이전트 프레임워크입니다.

### 핵심 기능
1. **복잡한 작업 처리**: 다단계 작업을 계획하고 분해할 수 있습니다
2. **파일 시스템 도구**: 대량의 컨텍스트를 파일로 관리합니다
3. **Subagent 위임**: 특수화된 하위 에이전트에게 작업을 위임하여 컨텍스트를 격리합니다
4. **영구 메모리**: 대화 및 스레드 간 메모리를 유지합니다

### DeepAgents vs LangGraph

`DeepAgents`는 `LangGraph`보다 더 자동화된 에이전트 프레임워크이고, `LangGraph`는 `workflow 설계` 중심 에이전트 프레임워크입니다.

DeepAgents는 
> "내가 알아서 파일 저장도 하고, 계획도 세우고, sub-agent도 만들고,     
> 컨텍스트 정리도 하고, 작업을 쪼개서 처리할게.     
> LangGraph처럼 사람이 DAG 설계하지 않아도 돼."     

반대로 LangGraph는:
> "네가 설계해라.    
> 노드/엣지/상태 흐름을 네가 제어해라.    
> 난 framework일 뿐이야."    

| 기능           | DeepAgents (내장)                  | LangGraph            |
| ------------ | -------------------------------- | -------------------- |
| 상태 관리(State) | 자체 backend / store 지원            | DAG 기반 state machine |
| 멀티스텝 작업      | Todo + subagent + 계획 기능 내장       | DAG 노드간 메세지 전달       |
| 컨텍스트 관리      | 자동 요약 / 파일 저장 / context eviction | 그래프 노드의 state 유지     |
| Worker 분리    | subagents                        | 여러 agent 노드 구성       |
| 롱런 작업 관리     | harness가 자동 처리                   | 그래프체크포인트로 안정적 운영     |


### DeepAgents vs Swarm vs LangGraph


| 항목            | Swarm             | DeepAgents               | LangGraph           |
| ------------- | ----------------- | ------------------------ | ------------------- |
| 철학            | 단순 agent routing  | 자동화된 오토에이전트              | DAG 기반 workflow     |
| 상태 관리         | 거의 없음             | strong (harness)         | 매우 strong           |
| 복잡도           | 매우 낮음             | 중간                       | 높음                  |
| 확장성           | 낮음                | 중간~높음                    | 매우 높음               |
| 용도            | Agent2Agent 프로토타입 | 파일/메모리/하위작업 포함 자동 AI 작업기 | 엔터프라이즈급 멀티 에이전트 시스템 |


## 2. 환경 설정

### Prerequisites
- Python 3.9 이상
- API 키 (Anthropic, OpenAI 등)


In [ ]:
# 필요한 패키지 설치
%pip install -q deepagents tavily-python


In [ ]:
# 환경 변수 설정
import os
from getpass import getpass

# API 키 설정
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")


## 3. 첫 번째 Deep Agent 만들기

### 기본 Agent 생성

`create_agent()` 함수를 사용하면 간단하게 Deep Agent를 생성할 수 있습니다.


In [ ]:
from deepagents import create_agent

# 기본 설정으로 Agent 생성
agent = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="당신은 도움이 되는 연구 어시스턴트입니다. 사용자의 질문에 답하기 위해 조사하고 보고서를 작성합니다."
)

print("✅ Deep Agent가 성공적으로 생성되었습니다!")


## 4. Agent 실행하기

### 간단한 질문으로 테스트


In [ ]:
# Agent에게 간단한 작업 요청
response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! Deep Agents에 대해 간단히 설명해주세요."}]},
    config={"configurable": {"thread_id": "demo-1"}}
)

# 응답 출력
print("\n🤖 Agent 응답:")
print(response["messages"][-1].content)


### 복잡한 연구 작업 수행하기

Deep Agent는 복잡한 다단계 작업을 수행할 수 있습니다.


In [ ]:
# 연구 작업 요청
research_task = """
AI 에이전트의 최신 트렌드에 대해 조사하고, 
다음 내용을 포함한 간단한 보고서를 작성해주세요:
1. 주요 트렌드 3가지
2. 각 트렌드의 핵심 특징
3. 향후 전망
"""

response = agent.invoke(
    {"messages": [{"role": "user", "content": research_task}]},
    config={"configurable": {"thread_id": "demo-research"}}
)

print("\n📊 연구 결과:")
print(response["messages"][-1].content)


## 5. Agent의 실행 과정 살펴보기

Deep Agent가 작업을 어떻게 수행하는지 단계별로 확인할 수 있습니다.


In [ ]:
# 스트리밍 모드로 실행 과정 확인
task = "Python으로 간단한 계산기 프로그램을 만드는 방법을 설명해주세요."

print("🔄 Agent 실행 과정:\n")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": task}]},
    config={"configurable": {"thread_id": "demo-stream"}},
    stream_mode="updates"
):
    # 각 단계의 출력 표시
    for node_name, node_output in chunk.items():
        print(f"\n📍 노드: {node_name}")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                if hasattr(msg, 'content') and msg.content:
                    print(f"   내용: {msg.content[:100]}...") if len(str(msg.content)) > 100 else print(f"   내용: {msg.content}")


## 6. 기본 파일 시스템 활용

Deep Agent는 내장된 파일 시스템을 사용하여 정보를 저장하고 관리할 수 있습니다.


In [ ]:
# 파일 생성 및 관리 작업
file_task = """
다음 작업을 수행해주세요:
1. 'notes.txt'라는 파일을 만들고 "Deep Agents 학습 시작"이라고 작성
2. 파일 내용을 읽어서 확인
3. 파일 목록을 보여주세요
"""

response = agent.invoke(
    {"messages": [{"role": "user", "content": file_task}]},
    config={"configurable": {"thread_id": "demo-files"}}
)

print("\n📁 파일 시스템 작업 결과:")
print(response["messages"][-1].content)


## 7. Agent 상태 확인하기

Agent의 현재 상태를 확인할 수 있습니다.


In [ ]:
# Agent 상태 가져오기
state = agent.get_state(config={"configurable": {"thread_id": "demo-1"}})

print("\n📋 Agent 상태 정보:")
print(f"메시지 수: {len(state.values.get('messages', []))}")
print(f"다음 노드: {state.next}")
print(f"작업 완료: {len(state.next) == 0}")


## 8. 실습: 간단한 연구 Assistant 만들기

배운 내용을 활용하여 특정 주제에 대해 조사하고 보고서를 작성하는 Agent를 만들어봅시다.


In [ ]:
# 연구 전용 Agent 생성
research_agent = create_agent(
    model="claude-3-5-sonnet-20241022",
    system_prompt="""
    당신은 전문 연구 어시스턴트입니다. 
    주어진 주제에 대해:
    1. 온라인에서 최신 정보를 검색합니다
    2. 수집한 정보를 분석하고 정리합니다
    3. 구조화된 보고서를 작성합니다
    4. 보고서를 파일로 저장합니다
    
    항상 출처를 명시하고, 객관적이고 정확한 정보를 제공하세요.
    """
)

print("✅ 연구 전용 Agent가 생성되었습니다!")


In [ ]:
# 연구 작업 실행
research_topic = """
RAG (Retrieval-Augmented Generation)의 최신 발전 동향에 대해 조사하고,
'rag_report.txt' 파일에 다음 내용을 포함한 보고서를 작성해주세요:

1. RAG 기술 개요
2. 최신 개선 방법 3가지
3. 실제 적용 사례
4. 향후 전망
"""

print("🔍 연구를 시작합니다...\n")

response = research_agent.invoke(
    {"messages": [{"role": "user", "content": research_topic}]},
    config={"configurable": {"thread_id": "research-rag"}}
)

print("\n✅ 연구 완료!")
print(response["messages"][-1].content)


## 9. 주요 개념 정리

### Deep Agent의 핵심 구성 요소

1. **create_agent()**: Agent 생성 함수
   - `model`: 사용할 LLM 모델 지정
   - `system_prompt`: Agent의 역할과 행동 정의

2. **invoke()**: 단일 작업 실행
   - 동기적으로 작업을 실행하고 결과 반환
   - `thread_id`로 대화 컨텍스트 유지

3. **stream()**: 실시간 스트리밍 실행
   - Agent의 실행 과정을 단계별로 확인
   - 중간 결과를 실시간으로 받을 수 있음

4. **파일 시스템**: 내장 파일 관리 도구
   - 파일 생성, 읽기, 쓰기, 삭제 가능
   - 대량의 컨텍스트 정보를 효율적으로 관리


## 10. 다음 단계

이제 Deep Agent의 기본을 이해했습니다! 다음 강의에서는:

- ✅ **Subagents & Workflow**: 복잡한 작업을 모듈화하고 여러 Agent가 협력하는 방법
- ✅ **Backend & Persistent Storage**: 영구 메모리와 상태 관리
- ✅ **Middleware & 고급 설정**: 커스텀 동작과 고급 기능

를 배우게 됩니다.

---

## 💡 연습 문제

### 문제 1: 날씨 조사 Agent
특정 도시의 날씨를 조사하고 여행 추천을 제공하는 Agent를 만들어보세요.

### 문제 2: 코드 작성 Assistant
Python 코드를 작성하고 파일로 저장하는 Agent를 만들어보세요.

### 문제 3: 멀티턴 대화
같은 thread_id를 사용하여 이전 대화 내용을 기억하는 Agent와 대화해보세요.

---

## 📚 참고 자료
- [Deep Agents 공식 문서](https://docs.langchain.com/oss/python/deepagents/overview)
- [LangGraph 문서](https://langchain-ai.github.io/langgraph/)
- [LangChain 문서](https://python.langchain.com/)
